# PyTorch Training Skeleton - Put the jobs back in working memory

You remember neural networks, but the five lines of a PyTorch update are
easy to blur together after time away. Which one guesses, which one
writes feedback, and which one actually changes a weight?

Keep a small **calm/alert sensor classifier**: eight hand-written
two-reading examples and two classes. We can inspect every answer without
turning a refresher into another training project.

| Need a reminder of | Look for |
|---|---|
| Pairing and delivery | Dataset items versus four-row batches |
| Stored state | Registered layers versus their activations |
| Learning | Clear, predict, score, backward, step |
| Reporting | Example-weighted loss and training accuracy |
| Using the result | Eval mode plus inference mode |

We'll run this core loop and check a tiny training fit. There is **no
validation, device branch, scheduler, mixed precision, or checkpoint
recovery** in this shortcut. Parts 01–05 supply those missing pieces;
the closing route table points to each one.
[Handwritten theory](pytorch-training-skeleton-theory.md).


## 1 - Keep a reading attached to its answer

The first four points have negative readings and target 0, calm.
The last four have positive readings and target 1, alert. For example,
`[-1.2,-0.8]` is paired with 0 and `[0.8,1.1]` with 1.
These are practice points, not a real alarm or a held-out safety test.

`TensorDataset` binds each input row to its integer target.
The loader shuffles row addresses and groups four at a time:

```text
all inputs [8,2], targets [8] -> batch inputs [4,2], targets [4]
```

The target `1` names the alert score column; it isn't a one-hot vector
or another floating-point feature.

![The explicit loop clears, predicts and measures, backpropagates, then updates.](../optional/framework-translation/images/3.png)

> **Visual to add — Shuffle sealed sensor cards**
> Show the eight actual input pairs from the next cell with their calm/alert
> IDs sealed onto each card, then two shuffled groups of four. Keep two
> measurements within each card; never shuffle features or labels
> independently. Suggested caption: "Delivery can reorder examples, not
> their checking answers."

The next cell prints a real batch. Taking the preview advances its loader
generator; it isn't a promise that training's first batch repeats it.


In [ ]:
import sys
print(f"Kernel interpreter: {sys.executable}")

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 7
torch.manual_seed(SEED)

inputs = torch.tensor(
    [
        [-1.2, -0.8],
        [-1.0, -1.3],
        [-0.7, -0.9],
        [-1.3, -1.1],
        [0.8, 1.1],
        [1.2, 0.7],
        [0.9, 1.4],
        [1.3, 1.0],
    ],
    dtype=torch.float32,
)
targets = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1], dtype=torch.long)

# Inputs (8 examples, 2 features) and targets (8,) stay paired by row;
# the loader returns (4, 2)/(4,), grouping examples without changing columns.
dataset = TensorDataset(inputs, targets)
loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

batch_inputs, batch_targets = next(iter(loader))
print("Full data:", tuple(inputs.shape), tuple(targets.shape))
print("One batch:", tuple(batch_inputs.shape), tuple(batch_targets.shape))
assert batch_inputs.shape == (4, 2)
assert batch_targets.dtype == torch.long

## 2 - Build a guesser, not a training command

`TinyClassifier` creates two layers once. For four current input rows,
the calculation is `[4,2] -> [4,8] -> [4,2]`.
ReLU bends the eight hidden features without adding rows or parameters.

The model stores **42 parameters**: `2×8 + 8 + 8×2 + 2`.
The four-row hidden tensor holds **32 activations**. Those are results for
this input, not another 32 stored trainable numbers.

`super().__init__()` enables registration, so `model.parameters()` can
find the weights and biases. `forward` describes one prediction.
Call `model(inputs)` to keep the module's usual hooks around that path.
Neither `forward` nor its call calculates a task loss or starts practice.

The last columns are raw **calm/alert logits**. Leave them raw for
cross-entropy; the correct integer ID chooses which column is right.
The next cell prints parameter names and checks the two-score output shape.


In [ ]:
class TinyClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # (B examples, input_size=2) -> (B, hidden_size=8) -> (B, output_size=2).
        # ReLU keeps hidden coordinates but adds nonlinearity; the last columns
        # are raw class scores, not another batch axis or probabilities.
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, inputs):
        return self.network(inputs)


model = TinyClassifier(input_size=2, hidden_size=8, output_size=2)
logits = model(batch_inputs)

print("Batch inputs:", tuple(batch_inputs.shape))
print("Batch logits:", tuple(logits.shape))
print("Trainable tensors:", [name for name, _ in model.named_parameters()])
assert logits.shape == (len(batch_inputs), 2)

## 3 - Read the loop as five different state changes

```text
zero_grad   remove old gradient feedback
forward     use current parameters to compute score activations
loss        compare each score row with its correct class ID
backward    add feedback to parameter .grad buffers
step        let the optimizer change parameters
```

Adam also remembers running gradient summaries. That optimizer memory isn't
an input, an activation, or a parameter owned by the model's forward path.
Clearing gradients doesn't erase Adam's memory or reset learned weights.

The next function receives the model, loader, loss, optimizer, and epoch
budget rather than hiding those choices inside the model. It returns
**history**; the trained parameters remain in the model object it received.

For each epoch it multiplies each batch's mean loss by its row count,
adds those totals, then divides by dataset size. That stays correct if
the final group is smaller. Here two batches of four make two updates
per epoch.

> **Visual to add — Five jobs, three kinds of persistent state**
> Use one calm/alert batch `[4,2]` and its four IDs. Keep the classifier's
> 42 parameter knobs, matching gradient notepads, and Adam's separate
> memory pad visually distinct. Follow scores `[4,2]` into one loss,
> backward into the notepads, and step into moved knobs. Show clearing
> affecting only the gradient notes. Suggested caption: "A score is a
> result, a gradient is advice, and a step is an action."

Defining `fit` doesn't run it. The following call supplies the choices.


In [ ]:
def fit(model, dataloader, loss_function, optimizer, epochs):
    history = []

    for _ in range(epochs):
        model.train()
        total_loss = 0.0

        for batch_inputs, batch_targets in dataloader:
            optimizer.zero_grad()
            logits = model(batch_inputs)
            loss = loss_function(logits, batch_targets)
            loss.backward()
            optimizer.step()

            total_loss += loss.item() * len(batch_inputs)

        average_loss = total_loss / len(dataloader.dataset)
        history.append(average_loss)

    return history

## 4 - Wire it once, then inspect what it actually proved

Choose cross-entropy for integer targets and Adam at rate 0.03.
Eighty epochs over two batches make **160 updates**. The returned history
averages the losses observed as weights changed within each epoch; it
isn't a sequence of fresh end-of-epoch evaluations.

After practice, `model.eval()` selects prediction-time layer behavior and
`inference_mode()` skips graph tracking. This model has no mode-sensitive
layers, but the switches keep their separate jobs visible.

`argmax(dim=1)` chooses one of the two score columns per input row,
so `[8,2]` scores become `[8]` IDs. It doesn't need softmax to find the
largest score.

The next cell checks falling recorded loss, all eight **training** labels
correct, and no inference graph. Fitting eight supplied points is a useful
smoke test; it says nothing yet about unseen sensor readings.


In [ ]:
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
history = fit(
    model,
    loader,
    loss_function,
    optimizer,
    epochs=80,
)

model.eval()
with torch.inference_mode():
    final_logits = model(inputs)
    # (8 examples, 2 class scores) -> (8,): select one class ID per row.
    predictions = final_logits.argmax(dim=1)

# Eight correctness flags -> scalar mean: this measures fit on practice rows only.
accuracy = (predictions == targets).float().mean().item()
print(f"Average loss: {history[0]:.4f} -> {history[-1]:.4f}")
print(f"Training accuracy: {accuracy:.0%}")
print("Predictions:", predictions.tolist())

assert history[-1] < history[0]
assert accuracy == 1.0
assert not final_logits.requires_grad

The loop can stay while the input meaning changes. In the later text
lessons, a token is a chosen text piece and its integer ID fetches a
trainable embedding row. Those rows become features for another
classifier-shaped prediction. That is a shape bridge, not text training
performed by these eight sensor points.


## 5 - Change the guesser without replacing the routine

The [word-objective chapter](../../03-language-model-foundations/03-language-models/03A-training-skeleton-and-word-objectives.ipynb)
uses CBOW to guess a center word from surrounding IDs:

| Here | CBOW |
|---|---|
| Measurement rows `[B,2]` | Context ID rows `[B,C]` |
| Hidden features | Lookup `[B,C,D]`, then average context to `[B,D]` |
| Two logits `[B,2]` | Vocabulary logits `[B,V]` |
| Correct class ID `[B]` | Correct center-word ID `[B]` |
| Cross-entropy and five-job update | The same responsibilities |

`B` counts examples, `C` context positions, `D` features per fetched row,
and `V` possible word answers. Token preparation and useful word objectives
belong to that later chapter, not this refresher.

### The short version

- Pair inputs and targets before shuffle.
- Modules own registered parameters; forward returns computed activations.
- Backward fills gradients; only the optimizer step acts on them.
- Count updates by batches, and average loss by examples.
- Eval mode and inference tracking are separate switches.
- Training accuracy on eight points isn't held-out quality.

### Go directly to the missing piece

| Gap | Resume here |
|---|---|
| Axes, dtypes, broadcasting, autograd | [Part 02](../02-tensors-and-autograd/02-tensors-and-autograd.ipynb) |
| Module ownership, loss shapes, optimizer order | [Part 03](../03-models-and-training-loops/03-models-and-training-loops.ipynb) |
| Dataset/DataLoader, saved weights, optimizer state | [Part 04](../04-data-checkpoints-and-hyperparameters/04-data-checkpoints-and-hyperparameters.ipynb) |
| Validation, selection, modes, scheduling, saved preprocessing | [Part 05](../05-modern-training-lifecycle/05-modern-training-lifecycle.ipynb) |

**Next on the shortcut route:** [Sequence Memory](../../03-language-model-foundations/01-sequence-memory/README.md),
or [Tokenization and Embeddings](../../03-language-model-foundations/02-tokenization-and-embeddings/README.md)
when recurrence isn't your current goal.


## Copy/paste memory card

This is the same executable contract with the teaching output removed:

```python
class Model(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        # (B rows, input_size) -> (B, hidden_size) -> (B, output_size classes).
        # ReLU keeps the hidden shape but adds a bend; keep final scores raw
        # because CrossEntropyLoss handles their normalization.
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, output_size),
        )

    def forward(self, inputs):
        return self.network(inputs)


def fit(model, dataloader, loss_function, optimizer, epochs):
    history = []
    for _ in range(epochs):
        model.train()
        total_loss = 0.0
        for inputs, targets in dataloader:
            optimizer.zero_grad()
            predictions = model(inputs)
            loss = loss_function(predictions, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(inputs)
        history.append(total_loss / len(dataloader.dataset))
    return history


dataset = TensorDataset(inputs, targets)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)
model = Model(input_size, hidden_size, output_size)
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
history = fit(model, dataloader, loss_function, optimizer, epochs=10)

model.eval()
with torch.inference_mode():
    predictions = model(new_inputs)
```

### Coverage ledger

**Built and measured:** `TensorDataset`, `DataLoader`, `nn.Module`, `__init__`, `forward`, logits, cross-entropy, Adam, a reusable `fit`, correctly aggregated epoch loss, evaluation, and inference.

**Explained or linked:** parameter registration, optimizer state, classification-to-CBOW shape mapping, and the full-route handoff.

**Named and out of scope:** device placement, validation, checkpoints, schedulers, mixed precision, clipping, and distributed training. Part 05 covers validation, saved state, scheduling, and the CUDA mixed-precision boundary; it does not implement clipping or distributed training.